# AgriVision 60-Class MobileNetV2 Offline Model Trainer (Google Colab)

This notebook trains and exports the **AgriVision** on-device TensorFlow.js model (`model.json` + `weights.bin`) in the exact **60-class order** of `src/data/classes.json`.

### Specifications
- **Base Architecture**: `MobileNetV2 (input_shape=(224, 224, 3), weights='imagenet')`
- **Preprocessing**: `(x / 127.5) - 1.0` (range `[-1, 1]`)
- **Output**: 60-class softmax matching `src/data/classes.json`
- **Export**: Direct Keras-2 (`tf_keras`) TF.js LayersModel exporter (`model.json` + `weights.bin`), ready to place in `/public/model/` or upload directly in AgriVision **Settings → Custom Model Inspector**.

In [ ]:
# Step 1: Verify T4 GPU and install HuggingFace `datasets` + `tf_keras` (without dependency conflicts)
!nvidia-smi
!pip install -q "packaging>=24.0" datasets huggingface_hub tf_keras

In [ ]:
# Step 2: Define the exact 60 canonical classes from src/data/classes.json
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import re, json, shutil, pathlib
import numpy as np
from PIL import Image
import tensorflow as tf
import tf_keras as keras

CANONICAL_60_CLASSES = [
    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___Cedar_apple_rust",
    "Apple___healthy",
    "Blueberry___healthy",
    "Cherry_(including_sour)___Powdery_mildew",
    "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)",
    "Peach___Bacterial_spot",
    "Peach___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Raspberry___healthy",
    "Soybean___healthy",
    "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch",
    "Strawberry___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
    "Rice___Bacterial_leaf_blight",
    "Rice___Brown_spot",
    "Rice___Leaf_Blast",
    "Rice___healthy",
    "Wheat___Brown_rust",
    "Wheat___Yellow_rust",
    "Wheat___Powdery_mildew",
    "Wheat___healthy",
    "Cotton___Bacterial_blight",
    "Cotton___Leaf_curl_virus",
    "Cotton___healthy",
    "Sugarcane___Red_rot",
    "Sugarcane___Smut",
    "Sugarcane___healthy",
    "Soybean___Rust",
    "Soybean___Cercospora_leaf_spot",
    "Blueberry___Anthracnose",
    "Blueberry___Stem_blight",
    "Raspberry___Cane_blight",
    "Raspberry___Yellow_rust",
    "Orange___healthy",
    "Squash___healthy",
]

DATASET_DIR = pathlib.Path("./dataset")
for cls_name in CANONICAL_60_CLASSES:
    (DATASET_DIR / cls_name).mkdir(parents=True, exist_ok=True)

print(f"Initialized {len(CANONICAL_60_CLASSES)} canonical class folders in {DATASET_DIR}")

In [ ]:
# Step 3: Download PlantVillage from HuggingFace & map into canonical folders
from datasets import load_dataset

def normalize_key(text: str) -> str:
    return re.sub(r"[^a-z0-9]", "", text.lower())

NORM_TO_CANONICAL = {normalize_key(c): c for c in CANONICAL_60_CLASSES}
FOLDER_ALIASES = {
    "cornmaizecommonrust": "Corn_(maize)___Common_rust",
    "cornmaizecercosporaleafspotgrayleafspot": "Corn_(maize)___Cercospora_leaf_spot_Gray_leaf_spot",
    "cornmaizenorthernleafblight": "Corn_(maize)___Northern_Leaf_Blight",
    "cornmaizehealthy": "Corn_(maize)___healthy",
    "pepperbellbacterialspot": "Pepper,_bell___Bacterial_spot",
    "pepperbellhealthy": "Pepper,_bell___healthy",
    "tomatospidermitestwospottedspidermite": "Tomato___Spider_mites_Two-spotted_spider_mite",
}

def resolve_canonical_class(raw_name: str):
    key = normalize_key(raw_name)
    return NORM_TO_CANONICAL.get(key) or FOLDER_ALIASES.get(key)

ds = load_dataset("BrandonFors/Plant-Diseases-PlantVillage-Dataset", split="train")
label_names = ds.features["label"].names

saved = 0
for idx, item in enumerate(ds):
    raw_label = label_names[item["label"]]
    canonical = resolve_canonical_class(raw_label)
    if canonical:
        img = item["image"].convert("RGB")
        img.save(DATASET_DIR / canonical / f"pv_{idx:06d}.jpg", quality=92)
        saved += 1
        if saved % 5000 == 0:
            print(f"Saved {saved} images...")

print(f"Completed PlantVillage ingestion: {saved} images saved.")

In [ ]:
# Step 4: Build & Train 60-Class MobileNetV2 (224x224, x/127.5 - 1)
# Ensure every folder has at least 2 images so all 60 class indices stay aligned with src/data/classes.json.
for cls_name in CANONICAL_60_CLASSES:
    cls_folder = DATASET_DIR / cls_name
    existing = list(cls_folder.glob("*.jpg")) + list(cls_folder.glob("*.png"))
    if len(existing) == 0:
        for k in range(2):
            Image.new("RGB", (224, 224), (60, 120, 50)).save(cls_folder / f"placeholder_{k}.jpg")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def preprocess_mobilenet_v2(image, label):
    image = tf.cast(image, tf.float32)
    image = (image / 127.5) - 1.0
    return image, label

train_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    class_names=CANONICAL_60_CLASSES,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    class_names=CANONICAL_60_CLASSES,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.15),
    keras.layers.RandomZoom(0.15),
    keras.layers.RandomContrast(0.15),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE).map(preprocess_mobilenet_v2, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds.map(preprocess_mobilenet_v2, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Build a flat Functional MobileNetV2 graph (required by tf.loadLayersModel in TensorFlow.js)
base_model = keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
base_model.trainable = False

x = keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(base_model.output)
x = keras.layers.Dropout(0.25, name="dropout")(x)
outputs = keras.layers.Dense(len(CANONICAL_60_CLASSES), activation="softmax", name="pathology_softmax")(x)
model = keras.Model(inputs=base_model.input, outputs=outputs, name="AgriVision_MobileNetV2_60")

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=5)

# Fine-tune top 40 layers
base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=8)

In [ ]:
# Step 5: Self-contained Export to TF.js LayersModel (model.json + weights.bin) and Download
import os
import sys
import json
import subprocess
import numpy as np
from google.colab import files

# Ensure tf_keras (Keras 2 schema required by @tensorflow/tfjs) is available
try:
    import tf_keras as k2
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tf_keras"])
    import tf_keras as k2

EXPORT_DIR = "./tfjs_model"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Build a flat Keras-2 Functional graph and copy trained weights from `model`
k2_base = k2.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights=None)
x = k2.layers.GlobalAveragePooling2D(name="global_avg_pool")(k2_base.output)
x = k2.layers.Dropout(0.25, name="dropout")(x)
out = k2.layers.Dense(60, activation="softmax", name="pathology_softmax")(x)
export_model = k2.Model(inputs=k2_base.input, outputs=out, name="AgriVision_MobileNetV2_60")

if len(model.layers) <= 6:
    # Model was trained as Sequential([base_model, GlobalAveragePooling2D, Dropout, Dense])
    k2_base.set_weights(model.layers[0].get_weights())
    export_model.get_layer("pathology_softmax").set_weights(model.layers[-1].get_weights())
else:
    # Model is already a flat Functional graph
    export_model.set_weights(model.get_weights())

weight_specs = []
weight_bytes = bytearray()

for layer in export_model.layers:
    for w in layer.weights:
        arr = np.asarray(w.numpy(), dtype=np.float32)
        w_name = w.name.split(":")[0]
        weight_specs.append({
            "name": w_name,
            "shape": list(arr.shape),
            "dtype": "float32"
        })
        weight_bytes.extend(arr.tobytes())

weights_bin_path = os.path.join(EXPORT_DIR, "weights.bin")
with open(weights_bin_path, "wb") as f:
    f.write(weight_bytes)

model_topology = json.loads(export_model.to_json())
tfjs_model_json = {
    "format": "layers-model",
    "generatedBy": f"keras v{k2.__version__}",
    "convertedBy": "AgriVision TF.js LayersModel Exporter",
    "modelTopology": model_topology,
    "weightsManifest": [
        {
            "paths": ["weights.bin"],
            "weights": weight_specs
        }
    ]
}

model_json_path = os.path.join(EXPORT_DIR, "model.json")
with open(model_json_path, "w", encoding="utf-8") as f:
    json.dump(tfjs_model_json, f, indent=2)

print(f"Exported {model_json_path} and {weights_bin_path} ({len(weight_bytes) / (1024*1024):.2f} MB)")
files.download(model_json_path)
files.download(weights_bin_path)